# Lifecycle Hooks - Monitor Agent Execution

## Purpose
Learn how to monitor and instrument agent execution using lifecycle hooks. Hooks provide visibility into agent behavior for logging, metrics collection, debugging, and performance monitoring.

## Key Concepts
- **RunHooks**: Base class for defining custom lifecycle hooks
- **Event Callbacks**: Methods called at specific points in execution
- **on_agent_start**: Triggered when an agent begins processing
- **on_llm_end**: Called after each LLM response
- **on_agent_end**: Invoked when agent completes execution

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `RunHooks` to create custom lifecycle hooks:

In [ ]:
import asyncio
from agents import Agent, Runner, RunHooks
from dataclasses import dataclass

## Step 1: Define Custom Hooks Class

Create a subclass of `RunHooks` and implement callback methods:

**Available Hooks**:
- `on_agent_start(context, agent)` - Called when agent starts
- `on_llm_end(context, agent, response)` - Called after each LLM response
- `on_agent_end(context, agent, output)` - Called when agent finishes
- `on_tool_start(context, agent, tool, input)` - Before tool execution
- `on_tool_end(context, agent, tool, output)` - After tool execution

💡 **Async Methods**: All hooks should be `async def` functions.

🔍 **Access**: Hooks receive `context` with usage stats, timing, and state.

In [ ]:
class LoggingHooks(RunHooks):
    async def on_agent_start(self, context, agent):
        print(f"Starting {agent.name}")

    async def on_llm_end(self, context, agent, response):
        print(f"{agent.name} produced {len(response.output)} output items")

    async def on_agent_end(self, context, agent, output):
        print(f"{agent.name} finished with usage: {context.usage}")

## Step 2: Create Agent

Create a regular agent - hooks are applied at execution time, not creation time:

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="Be concise",
    model="openai.gpt-5.5",
)

## Step 3: Run with Hooks

Pass hooks instance to `Runner.run()` to enable monitoring:

**Execution Flow**:
1. `on_agent_start()` called
2. Agent processes input
3. `on_llm_end()` called after each LLM response
4. `on_agent_end()` called with final output

🎯 **Result**: Watch the execution lifecycle in real-time!

In [ ]:
result = await Runner.run(
    agent, 
    input = "Explain quines",
     hooks=LoggingHooks())
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Lifecycle Hooks** notebook!